<a href="https://colab.research.google.com/github/yuliia-naumeniuk/ab-testing-statistical-analysis/blob/main/A_B_Testing_Statistical_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Connecting to the Database**

In [ ]:
# Installing a library to work with Google BigQuery
!pip install --upgrade google-cloud-bigquery

# Libraries
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import statsmodels.api as sm

# Authentication
auth.authenticate_user()

# Creating a BigQuery client
client = bigquery.Client(project="data-analytics-mate")

In [ ]:
# SQL query
query = """
WITH
 session_info AS (
   SELECT
     s.date,
     s.ga_session_id,
     sp.country,
     sp.device,
     sp.continent,
     sp.channel,
     ab.test,
     ab.test_group
   FROM `DA.ab_test` ab
   JOIN `DA.session` s
     ON ab.ga_session_id = s.ga_session_id
   JOIN `DA.session_params` sp
     ON sp.ga_session_id = ab.ga_session_id
 ),
 session_with_orders AS (
   SELECT
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group,
     COUNT(DISTINCT o.ga_session_id) AS session_with_orders
   FROM `DA.order` o
   JOIN session_info
     ON o.ga_session_id = session_info.ga_session_id
   GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group
 ),
 event AS (
   SELECT
     session_info.date,
     session_info.country,
    session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group,
     ep.event_name,
     COUNT(ep.ga_session_id) AS event_cnt
   FROM `DA.event_params` ep
   JOIN session_info
     ON ep.ga_session_id = session_info.ga_session_id
   GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group,
     ep.event_name
 ),
 session AS (
   SELECT
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group,
     COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
   FROM session_info
 GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group
 ),
 account AS (
   SELECT
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group,
     COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
   FROM `DA.account_session` acs
   JOIN session_info
     ON acs.ga_session_id = session_info.ga_session_id
   GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test,
     session_info.test_group
 )
SELECT
 session_with_orders.date,
 session_with_orders.country,
 session_with_orders.device,
 session_with_orders.continent,
 session_with_orders.channel,
 session_with_orders.test,
 session_with_orders.test_group,
 'session with orders' AS event_name,
 session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
 event.date,
 event.country,
 event.device,
 event.continent,
 event.channel,
 event.test,
 event.test_group,
 event_name,
 event_cnt AS value
FROM event
UNION ALL
SELECT
 session.date,
 session.country,
 session.device,
 session.continent,
 session.channel,
 session.test,
 session.test_group,
 'session' AS event_name,
 session_cnt AS value
FROM session
UNION ALL
SELECT
 account.date,
 account.country,
 account.device,
 account.continent,
 account.channel,
 account.test,
 account.test_group,
 'new account' AS event_name,
 new_account_cnt AS value
FROM account
  """

In [ ]:
# Executing the query
query_job = client.query(query)  # Executing the SQL query
results = query_job.result()  # Waiting for the query to complete


# Converting results to a DataFrame
df = results.to_dataframe()


# Displaying the results
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-02,North Macedonia,desktop,Europe,Direct,2,1,session with orders,1
1,2020-11-03,New Zealand,desktop,Oceania,Direct,2,2,session with orders,1
2,2020-11-04,Bulgaria,mobile,Europe,Paid Search,2,1,session with orders,1
3,2020-11-04,Kuwait,mobile,Asia,Organic Search,2,2,session with orders,1
4,2020-11-05,Serbia,desktop,Europe,Social Search,2,2,session with orders,1


# **2. Overall Test Significance**

In [ ]:
# Metrics setup
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new account"]

results = []

In [ ]:
# Aggregate Events by Test and Test Group
total_df = df.groupby(["test", "test_group", "event_name"])["value"].sum().reset_index()

total_df.head()

,test,test_group,event_name,value
0,1,1,add_payment_info,1988
1,1,1,add_shipping_info,3034
2,1,1,add_to_cart,1395
3,1,1,begin_checkout,3784
4,1,1,click,368


In [ ]:
# Iterate through Tests
for test in total_df["test"].unique():
  test_data = total_df[total_df["test"] == test]

  print(f"\n ---Test {test}--- ")


  # Iterate through Metrics
  for metric in metrics:

    control_value = test_data[(test_data["test_group"] == 1) & (test_data["event_name"] == metric)]["value"].iloc[0]

    control_sessions = test_data[(test_data["test_group"] == 1) & (test_data["event_name"] == "session")]["value"].iloc[0]

    test_value = test_data[(test_data["test_group"] == 2) & (test_data["event_name"] == metric)]["value"].iloc[0]

    test_sessions = test_data[(test_data["test_group"] == 2) & (test_data["event_name"] == "session")]["value"].iloc[0]


    # Z-Test calculation
    z_stat, p_value = sm.stats.proportions_ztest([test_value, control_value], [test_sessions, control_sessions])


    # Metric change calculation
    metric_change = ((test_value / test_sessions) - (control_value / control_sessions)) / (control_value / control_sessions) * 100


    # Save results
    results.append({
    "test_number": test,
    "metric": metric,
    "numerator_event": metric,
    "denominator_event": "session",
    "numerator_control": control_value,
    "denominator_control": control_sessions,
    "conversion_rate_control": control_value / control_sessions,
    "numerator_test": test_value,
    "denominator_test": test_sessions,
    "conversion_rate_test": test_value / test_sessions,
    "metric_change_%": metric_change,
    "z_stat": z_stat,
    "p_value": p_value,
    "significant": p_value < 0.05
})


 ---Test 1--- 

 ---Test 2--- 

 ---Test 3--- 

 ---Test 4--- 


# **3. Statistical Significance by Device**

In [ ]:
# Aggregate Events by Test, Test Group and Device
device_df = df.groupby(["test", "test_group", "device", "event_name"])["value"].sum().reset_index()

device_df.head()

,test,test_group,device,event_name,value
0,1,1,desktop,add_payment_info,1130
1,1,1,desktop,add_shipping_info,1711
2,1,1,desktop,add_to_cart,821
3,1,1,desktop,begin_checkout,2108
4,1,1,desktop,click,246


In [ ]:
# Iterate through Tests
for test in device_df["test"].unique():
  test_data = device_df[device_df["test"] == test]


  # Iterate through Devices
  for device in test_data["device"].unique():

        device_data = test_data[test_data["device"] == device]

        print(f"\n--- Test {test} | Device: {device} ---")


        # Iterate through Metrics
        for metric in metrics:

          control_value = device_data[(device_data["test_group"] == 1) & (device_data["event_name"] == metric)]["value"].iloc[0]

          control_sessions = device_data[(device_data["test_group"] == 1) & (device_data["event_name"] == "session")]["value"].iloc[0]

          test_value = device_data[(device_data["test_group"] == 2) & (device_data["event_name"] == metric)]["value"].iloc[0]

          test_sessions = device_data[(device_data["test_group"] == 2) & (device_data["event_name"] == "session")]["value"].iloc[0]


          # Z-Test calculation
          z_stat, p_value = sm.stats.proportions_ztest([test_value, control_value], [test_sessions, control_sessions])


          print(f"Metric: {metric} | p-value: {p_value}")


--- Test 1 | Device: desktop ---
Metric: add_payment_info | p-value: 0.007209741371215498
Metric: add_shipping_info | p-value: 0.0003357584652017076
Metric: begin_checkout | p-value: 2.9541814920943097e-06
Metric: new account | p-value: 0.41152570800346633

--- Test 1 | Device: mobile ---
Metric: add_payment_info | p-value: 0.0007006374762030875
Metric: add_shipping_info | p-value: 0.867065231958409
Metric: begin_checkout | p-value: 0.7009567356723287
Metric: new account | p-value: 0.13374756193786455

--- Test 1 | Device: tablet ---
Metric: add_payment_info | p-value: 0.04586782579263077
Metric: add_shipping_info | p-value: 0.0914639928431928
Metric: begin_checkout | p-value: 0.014906985506589771
Metric: new account | p-value: 0.8713411093173797

--- Test 2 | Device: desktop ---
Metric: add_payment_info | p-value: 0.06943375930883558
Metric: add_shipping_info | p-value: 0.07925229942804458
Metric: begin_checkout | p-value: 0.004507241652049557
Metric: new account | p-value: 0.4763559

# **4. Statistical Significance by Channel**

In [ ]:
# Aggregate Events by Test, Test Group and Channel
channel_df = df.groupby(["test", "test_group", "channel", "event_name"])["value"].sum().reset_index()

channel_df.head()

,test,test_group,channel,event_name,value
0,1,1,Direct,add_payment_info,392
1,1,1,Direct,add_shipping_info,664
2,1,1,Direct,add_to_cart,269
3,1,1,Direct,begin_checkout,823
4,1,1,Direct,click,79


In [ ]:
# Iterate through Tests
for test in channel_df["test"].unique():
  test_data = channel_df[channel_df["test"] == test]


  # Iterate through Channels
  for channel in test_data["channel"].unique():

        channel_data = test_data[test_data["channel"] == channel]

        print(f"\n--- Test {test} | Channel: {channel} ---")


        # Iterate through Metrics
        for metric in metrics:

          control_value = channel_data[(channel_data["test_group"] == 1) & (channel_data["event_name"] == metric)]["value"].iloc[0]

          control_sessions = channel_data[(channel_data["test_group"] == 1) & (channel_data["event_name"] == "session")]["value"].iloc[0]

          test_value = channel_data[(channel_data["test_group"] == 2) & (channel_data["event_name"] == metric)]["value"].iloc[0]

          test_sessions = channel_data[(channel_data["test_group"] == 2) & (channel_data["event_name"] == "session")]["value"].iloc[0]


          # Z-Test calculation
          z_stat, p_value = sm.stats.proportions_ztest([test_value, control_value], [test_sessions, control_sessions])


          print(f"Metric: {metric} | p-value: {p_value}")



--- Test 1 | Channel: Direct ---
Metric: add_payment_info | p-value: 2.7285656989392044e-06
Metric: add_shipping_info | p-value: 0.040295440378197364
Metric: begin_checkout | p-value: 0.0028210870432094437
Metric: new account | p-value: 0.37885982126254425

--- Test 1 | Channel: Organic Search ---
Metric: add_payment_info | p-value: 0.0001909045129292049
Metric: add_shipping_info | p-value: 0.024127398901353108
Metric: begin_checkout | p-value: 0.030621596905618862
Metric: new account | p-value: 0.9619193439402388

--- Test 1 | Channel: Paid Search ---
Metric: add_payment_info | p-value: 0.029451531682692326
Metric: add_shipping_info | p-value: 0.6050741043001047
Metric: begin_checkout | p-value: 0.7352929738262673
Metric: new account | p-value: 0.26258737187988956

--- Test 1 | Channel: Social Search ---
Metric: add_payment_info | p-value: 0.31206144881364284
Metric: add_shipping_info | p-value: 0.058208620675484775
Metric: begin_checkout | p-value: 0.20374842306540475
Metric: new ac

# **5. Statistical Significance by Continent**

In [ ]:
# Aggregate Events by Test, Test Group and Continent
continent_df = df.groupby(["test", "test_group", "continent", "event_name"])["value"].sum().reset_index()

continent_df.head()

,test,test_group,continent,event_name,value
0,1,1,(not set),add_payment_info,7
1,1,1,(not set),add_shipping_info,4
2,1,1,(not set),add_to_cart,3
3,1,1,(not set),begin_checkout,4
4,1,1,(not set),click,1


In [ ]:
# Iterate through Tests
for test in continent_df["test"].unique():
  test_data = continent_df[continent_df["test"] == test]


  # Iterate through Continents
  for continent in test_data["continent"].unique():

        continent_data = test_data[test_data["continent"] == continent]

        print(f"\n--- Test {test} | Continent: {continent} ---")


        # Iterate through Metrics
        for metric in metrics:

          control_value = continent_data[(continent_data["test_group"] == 1) & (continent_data["event_name"] == metric)]["value"].iloc[0]

          control_sessions = continent_data[(continent_data["test_group"] == 1) & (continent_data["event_name"] == "session")]["value"].iloc[0]

          test_value = continent_data[(continent_data["test_group"] == 2) & (continent_data["event_name"] == metric)]["value"].iloc[0]

          test_sessions = continent_data[(continent_data["test_group"] == 2) & (continent_data["event_name"] == "session")]["value"].iloc[0]


          # Z-Test calculation
          z_stat, p_value = sm.stats.proportions_ztest([test_value, control_value], [test_sessions, control_sessions])


          print(f"Metric: {metric} | p-value: {p_value}")



--- Test 1 | Continent: (not set) ---
Metric: add_payment_info | p-value: 0.48668867355025014
Metric: add_shipping_info | p-value: 0.02654503387641662
Metric: begin_checkout | p-value: 0.0002105143780896666
Metric: new account | p-value: 0.35683904415325407

--- Test 1 | Continent: Africa ---
Metric: add_payment_info | p-value: 0.3581959726668148
Metric: add_shipping_info | p-value: 0.14812541880677874
Metric: begin_checkout | p-value: 0.03089358445102105
Metric: new account | p-value: 0.6694076814720262

--- Test 1 | Continent: Americas ---
Metric: add_payment_info | p-value: 0.05923178099558561
Metric: add_shipping_info | p-value: 0.27942702228096183
Metric: begin_checkout | p-value: 0.27793953584638764
Metric: new account | p-value: 0.008826843842463641

--- Test 1 | Continent: Asia ---
Metric: add_payment_info | p-value: 0.5702422481017848
Metric: add_shipping_info | p-value: 0.21898733380776259
Metric: begin_checkout | p-value: 0.15248585784948884
Metric: new account | p-value: 0

# **6. Tableau Dashboard**

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)

results_df.head()

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_%,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,new account,new account,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False


In [ ]:
# Saving data for visualization in Tableau


# Excel file
from google.colab import drive

drive.mount("/content/drive")

results_df.to_excel("/content/drive/MyDrive/ab_test_results.xlsx", index=False)


# CSV file
results_df.to_csv("/content/drive/MyDrive/ab_test_results.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


A/B Test Results:

[Excel File](https://docs.google.com/spreadsheets/d/1myCgcQbqS0zzngGpj4sCiOrZLCN5jy82/edit?usp=drive_link&ouid=112977218335542423372&rtpof=true&sd=true)

[СSV File](https://drive.google.com/file/d/1OwODKHYD_JW_l_NlY6_S57f6gTgzNTUS/view?usp=drive_link)

Tableau Dashboard:

[View in Tableau](https://public.tableau.com/views/ABTest_17717709255380/ABTest?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

# **7. Overall Conclusions**


(EN)

**Test 1:**

A statistically significant positive effect was found for the metrics **add_payment_info**, **add_shipping_info**, and **begin_checkout** (p-value < 0.05).

The **new account** metric was not statistically significant.

**Test 2:**

For all analyzed metrics, the p-values are above 0.05.

This means that no statistically significant differences were found between the control group and the test group.

**Test 3:**

A statistically significant negative effect was found for the **begin_checkout** metric (p-value = 0.012 < 0.05).

The other metrics were not statistically significant.

**Test 4:**

A statistically significant negative effect was found for the **begin_checkout** metric (p-value = 0.046 < 0.05) and the **new account** metric (p-value = 0.018 < 0.05).

The **add_payment_info** and **add_shipping_info** metrics were not statistically significant.

(PL)

**Test 1:**

Wykryto statystycznie istotny pozytywny wpływ na metryki ***add_payment_info***, ***add_shipping_info*** oraz ***begin_checkout*** (p-value < 0,05).

Metryka ***new account*** nie wykazała istotności statystycznej.

**Test 2:**

Dla wszystkich analizowanych metryk wartości p-value przekraczają 0,05.

Oznacza to, że nie stwierdzono statystycznie istotnych różnic między grupą kontrolną a grupą testową.

**Test 3:**

Wykryto statystycznie istotny negatywny wpływ na metrykę ***begin_checkout*** (p-value = 0,012 < 0,05).

Pozostałe metryki nie wykazały istotności statystycznej.

**Test 4:**

Odnotowano statystycznie istotny negatywny wpływ na metryki ***begin_checkout*** (p-value = 0,046 < 0,05) oraz ***new account*** (p-value = 0,018 < 0,05).

Metryki ***add_payment_info*** oraz ***add_shipping_info*** pozostały statystycznie nieistotne.